In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
import rapids_singlecell as rsc
import cupy as cp
import anndata as ad
import pandas as pd
import numpy as np
import os
import cupyx.scipy.sparse as cusparse
import scipy.sparse as sp

In [ ]:
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator

rmm.reinitialize(
    managed_memory=True,  # Allows oversubscription
    pool_allocator=False,  # default is False
    devices=[0],  # GPU device IDs to register. By default registers only GPU 0.
)
cp.cuda.set_allocator(rmm_cupy_allocator)

In [ ]:
proseg_outputs = "path/to/your/proseg/output/directory"
global_clustering_outputs = "path/to/your/output/directory"
epithelial_clustering_step1_outputs = "path/to/your/another/output/directory"
epithelial_clustering_step2_outputs = "path/to/your/another/output/directory"

#Please make a list of Xenium bundle directory name like:
section_ids = ['output-XETG00130__0076861__Normal_lung__20251122__033042', 
               'output-XETG00130__0076861__tkbadeno1__20251122__033041', 
               'output-XETG00130__0076861__tkbadeno2__20251122__033042', 
               'output-XETG00130__0076861__tkbadeno3__20251122__033042', 
               'output-XETG00130__0076861__tkbadeno4__20251122__033042', 
               'output-XETG00130__0076861__tkbadeno5__20251122__033042', 
               'output-XETG00130__0076861__tkbadeno6__20251122__033042', 
               'output-XETG00130__0076861__tkbadeno7__20251122__033042', 
               'output-XETG00130__0076862__Normal_lung__20251122__033042', 
               'output-XETG00130__0076862__tkbadeno8__20251122__033042', 
               'output-XETG00130__0076862__tkbadeno9__20251122__033042', 
               'output-XETG00130__0076862__tkbadeno10__20251122__033042', 
               'output-XETG00130__0076862__tkbadeno11__20251122__033042', 
               'output-XETG00130__0076862__tkbadeno12__20251122__033042', 
               'output-XETG00130__0076862__tkbadeno13__20251122__033042', 
               'output-XETG00130__0076862__tkbadeno14__20251122__033042', 
               'output-XETG00130__0076921__Normal_lung__20251203__041512', 
               'output-XETG00130__0076921__tkbadeno15__20251203__041512', 
               'output-XETG00130__0076921__tkbadeno16__20251203__041512', 
               'output-XETG00130__0076921__tkbadeno17__20251203__041512', 
               'output-XETG00130__0076922__Normal_lung__20251203__041512', 
               'output-XETG00130__0076922__tkbadeno18__20251203__041512', 
               'output-XETG00130__0076922__tkbadeno19__20251203__041512', 
               'output-XETG00130__0076922__tkbadeno20__20251203__041512']

In [ ]:
h5ad_path = epithelial_clustering_step1_outputs + "/proseg_nocorrection_ep.h5ad"
integrated = sc.read_h5ad(h5ad_path)

In [ ]:
integrated.obs["slide"] = integrated.obs["batch"].str.split("__").str[0]
integrated.obs["slide"] = integrated.obs["slide"].astype("category")

In [ ]:
integrated.obs

In [ ]:
# Please confirm the following clusters contain epithelial cells.
keep = {"1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12", "13", "14", "15", "16", "17", "18", "19", "20", "21", "22", "23", "24", "25", "26", "27"}

mask = integrated.obs["leiden"].astype(str).isin(keep)

ep = integrated[mask].copy()

In [ ]:
ep

In [ ]:
ep.obs = ep.obs.drop(columns=["leiden"], errors="ignore")
ep.var = ep.var.iloc[:, 0:0]
ep.obsm.clear()
ep.varm.clear()
ep.obsp.clear()

for k in list(ep.uns.keys()):
    if any(x in k for x in ["pca", "neighbors", "leiden", "umap", "hvg"]):
        ep.uns.pop(k)

In [ ]:
ep

In [ ]:
ep.X = ep.layers['counts'].copy()

In [ ]:
if sp.issparse(ep.X):
    ep.X = ep.X.astype(np.float32)    
else:
    ep.X = ep.X.astype(np.float32, copy=False)

rsc.get.anndata_to_GPU(ep)

In [ ]:
rsc.pp.filter_cells(ep, min_counts=200)
rsc.pp.filter_cells(ep, min_genes=80)
rsc.pp.normalize_total(ep, target_sum=500)
rsc.pp.log1p(ep)

In [ ]:
rsc.pp.pca(ep, random_state=7, use_highly_variable=False)

In [ ]:
sc.pl.pca_variance_ratio(ep, log=True)

In [ ]:
vr = ep.uns["pca"]["variance_ratio"]
cum_vr = np.cumsum(vr)

pcs = np.arange(1, len(vr) + 1)

plt.figure(figsize=(6,4))
plt.plot(pcs, cum_vr, marker="o")
plt.xlabel("Principal Component")
plt.ylabel("Cumulative Explained Variance Ratio")
plt.title("Cumulative explained variance")
plt.grid(True)
plt.show()

In [ ]:
# Adjust the number of principal components for the following dimension reduction procedures
ep.obsm["X_pca_6"] = ep.obsm["X_pca"][:, :6]

In [ ]:
rsc.pp.neighbors(ep, n_neighbors=5, use_rep="X_pca_6", random_state=7)

In [ ]:

rsc.tl.leiden(ep, resolution=0.45, key_added="leiden", random_state=7)

In [ ]:
rsc.get.anndata_to_GPU(ep)
rsc.tl.umap(ep, min_dist=0.5, spread=1.0, random_state=7)
sc.pl.umap(ep, color=['batch'])
sc.pl.umap(ep, color=["leiden"])

In [ ]:
clusters = ep.obs['leiden'].astype(str)
colors = ep.uns['leiden_colors']

color_map = dict(zip(sorted(clusters.unique()), colors))
color_map

In [ ]:
seurat_colors = [
    "#F8766D", "#D89000", "#A3A500", "#39B600", "#00BF7D",
    "#00BFC4", "#00B0F6", "#619CFF", "#9590FF", "#C77CFF",
    "#E76BF3", "#FF62BC", "#FF61C3", "#FF9DA7", "#F564E3",
    "#00BA38", "#00C08B", "#00A9FF", "#0072B2", "#56B4E9",
    "#009E73", "#E69F00", "#CC79A7", "#999999", "#7CAE00",
    "#00BFC4", "#C77CFF", "#F8766D", "#00BA38", "#619CFF"
]

In [ ]:
seurat_color_map = {
    str(i): seurat_colors[i % len(seurat_colors)]
    for i in range(len(color_map))
}

In [ ]:
ep.obs["color"] = clusters.map(seurat_color_map)

In [ ]:
ep.obs["leiden"] = ep.obs["leiden"].astype("category")
cats = ep.obs["leiden"].cat.categories
ep.uns["leiden_colors"] = seurat_colors[:len(cats)]

In [ ]:
sc.pl.umap(ep, color=['batch'])
sc.pl.umap(ep, color=["leiden"])

In [ ]:
plot_name = "Epithelium_UMAP_Sample"

umap = sc.pl.umap(ep, color=['batch'], title="", show=False)

fig = plt.gcf()
fig.subplots_adjust(top=0.88)
fig.suptitle(plot_name, y=0.95, fontsize=20)

fig.savefig(epithelial_clustering_step2_outputs + "/" + plot_name + ".tiff", bbox_inches="tight", dpi=800)
plt.close(fig)

In [ ]:
plot_name = "Epithelium_UMAP_Leiden"

umap = sc.pl.umap(ep, color=['leiden'], title="", show=False)

fig = plt.gcf()
fig.subplots_adjust(top=0.88)
fig.suptitle(plot_name, y=0.95, fontsize=20)

fig.savefig(epithelial_clustering_step2_outputs + "/" + plot_name + ".tiff", bbox_inches="tight", dpi=800)
plt.close(fig)

In [ ]:
marker_genes = {
    "Epithelium": ["EPCAM", "CDH1", "KRT5", "KRT8"],
    "Endothelium": ["PECAM1", "ERG", "CD31", "CD34", "CLDN5"],
    "Fibroblast": ["COL1A1", "DCN", "LUM"],
    "CAF": ["COL11A1", "CXCL12", "FAP", "POSTN"], 
    "Smooth muscle": ["ACTA2", "MYH11", "TAGLN"],
    "Pericyte": ["LAMC3", "RGS5", "PDGFRB"],
    "Inflammatory cells": ["PTPRC"],
    "T/NK-cell": ["CD3E", "GZMA"],
    "B-cell": ["BANK1", "CXCR5"],
    "Plasma cell": ["MZB1", "TNFRSF17"],
    "Macrophage": ["MARCO", "MS4A4A", "CD68"],
    "Dendritic cell": ["GZMB", "CLEC10A"],
    "Mast cell": ["RGS13", "KIT"],
    "Granulocytes": ["TREM1", "APOE"],
    "Proliferation": ["MKI67", "TOP2A", "MCM2", "PCNA"]
}

In [ ]:
marker_genes_in_data = dict()
for ct, markers in marker_genes.items():
    markers_found = list()
    for marker in markers:
        if marker in ep.var.index:
            markers_found.append(marker)
    marker_genes_in_data[ct] = markers_found

In [ ]:
marker_genes_in_data

In [ ]:
population = [
    "Epithelium",
    "Endothelium",
    "Fibroblast",
    "CAF", 
    "Smooth muscle",
    "Pericyte",
    "Inflammatory cells",
    "T/NK-cell",
    "B-cell",
    "Plasma cell",
    "Macrophage",
    "Dendritic cell",
    "Mast cell",
    "Granulocytes",
    "Proliferation"
]

In [ ]:
canonical_markers = {
    ct: [m for m in ct_markers if m in ep.var.index]
    for ct, ct_markers in marker_genes.items()
    if ct in population
}

In [ ]:
rsc.get.anndata_to_CPU(ep)

plot_name = "Epithelium_check_leiden_dotplot"

dp = sc.pl.dotplot(
    ep, 
    groupby="leiden", 
    title=None, 
    var_names=canonical_markers, 
    standard_scale="var", 
    figsize=(15, 20), 
    var_group_rotation=45, 
    show=False)

fig = plt.gcf()
fig.subplots_adjust(top=0.88)
fig.suptitle(plot_name, y=0.95, fontsize=20)

fig.savefig(epithelial_clustering_step2_outputs + "/" + plot_name + ".tiff", bbox_inches="tight", dpi=500)
plt.close(fig)

cluster_counts = ep.obs["leiden"].value_counts()
cluster_ratios = cluster_counts / cluster_counts.sum() * 100
print(cluster_ratios)

In [ ]:
for ss in range(len(section_ids)):
    adata = ep[ep.obs['batch'] == section_ids[ss][18:-18]].copy()
    df = adata.obs[["cell_id", "leiden", "color"]].copy()
    df.columns = ["cell_id", "group", "color"]
    path = epithelial_clustering_step2_outputs + "/ep_" + section_ids[ss][18:-18] + "_leiden.csv"
    df.to_csv(path, index=False)

In [ ]:
path = epithelial_clustering_step2_outputs + "/ep_cluster_ratios.csv"
cluster_ratios.to_csv(path, index=True)

In [ ]:
h5ad_path = epithelial_clustering_step2_outputs + "/epithelium.h5ad"
ep.write_h5ad(h5ad_path)